# IMU Chronos Feature Extraction

Generate Chronos embedding features from the IMU motion table and persist them as `general_db_imu_chronos.csv` (the file consumed by `imu/scripts/grid_search.py`). For each (label, subject) pair, the embedding is the mean over both time and feature dimensions of the Chronos encoder output.

## Configuration

In [ ]:
# Adjust these to point at your data root.
INPUT_CSV = "general_db_imu.csv"
OUTPUT_CSV = "general_db_imu_chronos.csv"
OUTPUT_DIR = "CHRONOS_FEATURES"
CHRONOS_MODEL = "amazon/chronos-t5-small"


## Setup

In [ ]:
import pandas as pd
import numpy as np
import os, glob
from tqdm import tqdm


if not os.path.exists(OUTPUT_DIR):
    os.mkdir(OUTPUT_DIR)



## Load Chronos pipeline and source data

In [ ]:
import pandas as pd
import torch
from chronos import ChronosPipeline

pipeline = ChronosPipeline.from_pretrained(
    CHRONOS_MODEL,
    device_map="cuda",
    torch_dtype=torch.bfloat16,
)

df = pd.read_csv(INPUT_CSV)
print(df.shape)

"""
# context must be either a 1D tensor, a list of 1D tensors,
# or a left-padded 2D tensor with batch as the first dimension
context = torch.tensor(df["#Passengers"])
embeddings, tokenizer_state = pipeline.embed(context)"""

## Compute embeddings per (label, subject)

In [ ]:
db = []
subjects = []
labels = []

for unique_label in tqdm(df["label"].unique()):
    label_data = df[df["label"] == unique_label]
    for unique_subject in label_data["subject"].unique():
        subject_data = label_data[label_data["subject"] == unique_subject]
        raw_data = subject_data.drop(columns=["label", "subject"]).to_numpy()
        embeddings, state = pipeline.embed(torch.tensor(raw_data))
        features = embeddings.detach().to(torch.float32).numpy().mean(axis=0).mean(axis=0)
        db.append(features)
        subjects.append(unique_subject)
        labels.append(unique_label)


## Aggregate into a DataFrame

In [ ]:
db_np = np.array(db)

In [ ]:
db_np.shape

In [ ]:
agg = {"subject": subjects, "label": labels, **{ f"features_{i}":j for i,j in enumerate(np.array(db).T)}}

In [ ]:
new_db = pd.DataFrame(agg)

## Save and inspect

In [ ]:
new_db

In [ ]:
new_db.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

In [ ]:
new_db["label"].value_counts()*0.8